In [ ]:
import json
from pathlib import Path

import pandas as pd


def execute_selected_notebook_cells(notebook_path: Path, cell_indexes: list[int]) -> dict:
    with notebook_path.open("r", encoding="utf-8") as handle:
        notebook = json.load(handle)

    namespace: dict = {}
    code_cells = [cell for cell in notebook["cells"] if cell.get("cell_type") == "code"]

    for cell_index in cell_indexes:
        cell = code_cells[cell_index]
        code = "\n".join(cell.get("source", []))
        exec(compile(code, str(notebook_path), "exec"), namespace)

    return namespace


workspace_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
wind_path = workspace_root / "notebooks" / "Wind_Speed.ipynb"
homes_path = workspace_root / "notebooks" / "Homes_Usage.ipynb"

wind_namespace = execute_selected_notebook_cells(wind_path, [0, 1])
homes_namespace = execute_selected_notebook_cells(homes_path, [1, 2, 3, 4, 5])

windturbine_daily_df = wind_namespace["windturbine_daily_df"].copy()
simulation_df = homes_namespace["simulation_df"].copy()

print("Loaded data from notebooks/Wind_Speed.ipynb and notebooks/Homes_Usage.ipynb")
print(f"Wind rows: {len(windturbine_daily_df)}")
print(f"Homes rows: {len(simulation_df)}")

In [ ]:
wind_daily = windturbine_daily_df.reset_index().rename(columns={"index": "date"})
wind_daily["date"] = pd.to_datetime(wind_daily["date"]).dt.strftime("%Y-%m-%d")
wind_daily = wind_daily[["date", "Windturbine_Produced"]].rename(
    columns={"Windturbine_Produced": "Windturbine_Produced_kwh_day"}
)

homes_daily = simulation_df[["date", "homes_shared_energy_usage_kWh"]].copy()
homes_daily["date"] = pd.to_datetime(homes_daily["date"]).dt.strftime("%Y-%m-%d")
homes_daily = homes_daily.rename(
    columns={"homes_shared_energy_usage_kWh": "homes_shared_energy_usage_kwh_day"}
)

daily_df = (
    wind_daily.merge(homes_daily, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

if daily_df.empty:
    raise RuntimeError("No matching daily records between wind and homes datasets.")

daily_df.head()

In [ ]:
import time
from IPython.display import clear_output

BATTERY_CAPACITY_KWH = 100_000.0
BATTERY_STORAGE_KWH = 25_000.0
SECONDS_PER_DAY = 1.0

simulation_output_rows = []

print("Starting battery simulation for 2025 (1 day = 1 second)...")

for row in daily_df.itertuples(index=False):
    date = row.date
    wind_kwh_day = row.Windturbine_Produced_kwh_day
    homes_kwh_day = row.homes_shared_energy_usage_kwh_day

    if pd.isna(wind_kwh_day) or pd.isna(homes_kwh_day):
        continue

    wind_kwh_day = float(wind_kwh_day)
    homes_kwh_day = float(homes_kwh_day)

    if wind_kwh_day == 0.0:
        clear_output(wait=True)
        print("No_Energy_Produced")
        break

    if (wind_kwh_day + BATTERY_STORAGE_KWH) < homes_kwh_day:
        clear_output(wait=True)
        print("Energy_Ran_Out")
        break

    battery_before = BATTERY_STORAGE_KWH
    surplus = wind_kwh_day - homes_kwh_day

    if surplus >= 0:
        BATTERY_STORAGE_KWH = min(BATTERY_CAPACITY_KWH, BATTERY_STORAGE_KWH + surplus)
        energy_lost_day = max(0.0, battery_before + surplus - BATTERY_CAPACITY_KWH)
    else:
        BATTERY_STORAGE_KWH = max(0.0, BATTERY_STORAGE_KWH + surplus)
        energy_lost_day = 0.0

    battery_percentage = int(round((BATTERY_STORAGE_KWH / BATTERY_CAPACITY_KWH) * 100))

    daily_output = {
        "date": date,
        "Windturbine_Produced_kwh_day": round(wind_kwh_day, 2),
        "homes_shared_energy_usage_kwh_day": round(homes_kwh_day, 2),
        "Battery_Percentage": battery_percentage,
        "Energy_Lost_Day_kwh": round(float(energy_lost_day), 2),
    }
    simulation_output_rows.append(daily_output)

    clear_output(wait=True)
    print(
        f"date={daily_output['date']} | "
        f"Windturbine_Produced_kwh_day={daily_output['Windturbine_Produced_kwh_day']:.2f} | "
        f"homes_shared_energy_usage_kwh_day={daily_output['homes_shared_energy_usage_kwh_day']:.2f} | "
        f"Battery_Percentage={daily_output['Battery_Percentage']} | "
        f"Energy_Lost_Day_kwh={daily_output['Energy_Lost_Day_kwh']:.2f}"
    )

    time.sleep(SECONDS_PER_DAY)
else:
    clear_output(wait=True)
    print("Simulation complete.")

battery_simulation_df = pd.DataFrame(simulation_output_rows)
battery_simulation_df.head()